# Evaluate DeepSeek Pro SFT — epoch 5
Evaluates `sft_s1k_deepseek_pro/epoch_5` on **AIME 2024, AIME 2025, AIME 2026,
AMC 2023, and MATH-500**: one answer per problem, 630 problems in total.
Uses the existing English generation/scoring code and continuous GPU batching.
Defaults match the latest old-SFT comparison: **temperature 0**, budget forcing at
**16,384 thinking tokens**, and **4,096 answer tokens** including the injected cue.
Temperature and thinking cap are editable below. No minimum-thinking extension is used.

Set `TRAIN_ROOT` to the exact Drive folder printed by your Pro training notebook,
including its timestamp if you started a fresh run. Leave it blank to discover a
single completed Pro epoch-5 checkpoint; multiple matches require an explicit choice.
Results, raw generations, settings and console logs are saved within that training
folder. A completed baseline evaluation is not required.

Use the **RTX PRO 6000 Blackwell 96GB** runtime and the `HF_TOKEN` Colab secret.
Push the project implementation to GitHub main before setup. Run settings, setup,
checkpoint selection and preparation first. Smoke and full evaluation have separate
disabled switches; enable them explicitly. This notebook never trains a model.
Rerun checkpoint selection and preparation after changing settings.

In [ ]:
# @title Settings
REPO_URL = "https://github.com/seungjun-green/post-training-aime.git"
CODE_ROOT = "/content/lg-aime-stage1-eval"
TRAIN_ROOT = "" # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive"
EVAL_ENV = "/content/lg-eval-env"
CONFIG = "configs/stage1_sft_deepseek_pro.yaml"
MODEL_KIND = "sft"
TEMPERATURE = 0.0 # @param {type:"number"}
MAX_THINKING_TOKENS = 16384 # @param {type:"integer"}

In [ ]:
import os, subprocess, sys, json
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
if not Path(CODE_ROOT).exists():
    subprocess.check_call(["git", "clone", "--branch", "main", REPO_URL, CODE_ROOT])
def git(*args):
    return subprocess.check_output(["git", "-C", CODE_ROOT, *args], text=True).strip()
if git("remote", "get-url", "origin") != REPO_URL or git("status", "--porcelain"):
    raise ValueError("Use a clean checkout of the configured repository")
if git("branch", "--show-current") != "main":
    raise ValueError("Use a main checkout")
subprocess.check_call(["git", "-C", CODE_ROOT, "pull", "--ff-only", "origin", "main"])
print("Evaluation code commit:", git("rev-parse", "HEAD"))
if not (Path(CODE_ROOT) / "eval/budget_generation.py").is_file():
    raise FileNotFoundError("Push all evaluation updates to GitHub main, then rerun setup.")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv==0.11.22"])
subprocess.check_call([sys.executable, "scripts/setup_eval_runtime.py", "--venv", EVAL_ENV], cwd=CODE_ROOT)
sys.path.insert(0, CODE_ROOT)
from common.process import run_logged

In [ ]:
# @title Locate the completed Pro epoch-5 checkpoint
checkpoint_relative = Path("checkpoints/stage1/sft_s1k_deepseek_pro/epoch_5/stage1_checkpoint.json")
if not TRAIN_ROOT.strip():
    candidates = sorted(
        root for root in Path(DRIVE_ROOT).glob("LG-AIME-Stage1*")
        if (root / checkpoint_relative).is_file()
    )
    if len(candidates) != 1:
        print("Completed Pro training folders:")
        for root in candidates:
            print(root)
        raise ValueError(
            "Set TRAIN_ROOT in Settings to your Pro training folder, then rerun this cell. "
            "No unique completed Pro epoch-5 checkpoint was found."
        )
    TRAIN_ROOT = str(candidates[0])
marker_path = Path(TRAIN_ROOT) / checkpoint_relative
if not marker_path.is_file():
    raise FileNotFoundError(f"Pro epoch 5 is missing: {marker_path}. Check the training output folder.")
marker = json.loads(marker_path.read_text())
if marker["epoch"] != 5:
    raise ValueError(f"Not an epoch-5 checkpoint: {marker_path}")
BASELINE_ROOT = str(Path(TRAIN_ROOT) / "evaluation_deepseek_pro")
print("Pro epoch-5 checkpoint:", marker_path.parent)
print("Evaluation output folder:", BASELINE_ROOT)

## Prepare the Pro checkpoint and evaluation settings
Epoch 5 is selected by default. Confirm the printed checkpoint path points to the
Pro training run you want. Outputs are isolated within that training folder under
`evaluation_deepseek_pro/profiles/`, with separate paths for temperature, budget,
smoke and full evaluation. Completed problems can be resumed with unchanged settings.

In [ ]:
import yaml
from eval.profiles import load_profile, profile_root
INCLUDE_EARLIER_EPOCHS = False
if MODEL_KIND not in {"sft", "base"}:
    raise ValueError("MODEL_KIND must be sft or base")
cfg = yaml.safe_load((Path(CODE_ROOT) / CONFIG).read_text())
eval_config = yaml.safe_load((Path(CODE_ROOT) / "configs/eval_english.yaml").read_text())
execution = yaml.safe_load((Path(CODE_ROOT) / "configs/eval_execution.yaml").read_text())
EVAL_PROFILE = "sample1_budget" if MODEL_KIND == "sft" else "sample1"
sampling = {"temperature": TEMPERATURE, "top_p": 1.0 if TEMPERATURE == 0 else 0.7}
budget = None
if MODEL_KIND == "sft":
    if type(MAX_THINKING_TOKENS) is not int or not 0 < MAX_THINKING_TOKENS < eval_config["max_new_tokens"]:
        raise ValueError("MAX_THINKING_TOKENS must be a positive integer below the total token budget")
    budget = yaml.safe_load((Path(CODE_ROOT) / "configs/eval_profiles.yaml").read_text())[EVAL_PROFILE]["budget_forcing"]
    budget["max_reasoning_tokens"] = MAX_THINKING_TOKENS
    budget["answer_budget_tokens"] = eval_config["max_new_tokens"] - MAX_THINKING_TOKENS
resolved_config, _ = load_profile(EVAL_PROFILE, eval_config, execution,
                                 budget_override=budget, sampling_override=sampling)
EVAL_LABEL = f"{MODEL_KIND}_{EVAL_PROFILE}_temperature_{float(TEMPERATURE)}"
if budget:
    EVAL_LABEL += f"_thinking_{MAX_THINKING_TOKENS}_answer_{budget['answer_budget_tokens']}"
config_dir = Path(TRAIN_ROOT) / "eval_configs"
config_dir.mkdir(parents=True, exist_ok=True)
sampling_path = config_dir / (EVAL_LABEL + "_sampling.yaml")
sampling_path.write_text(yaml.safe_dump(sampling, sort_keys=False))
RUN_ARGS = ["--sampling_config", str(sampling_path)]
if budget:
    budget_path = config_dir / (EVAL_LABEL + "_budget.yaml")
    budget_path.write_text(yaml.safe_dump(budget, sort_keys=False))
    RUN_ARGS += ["--budget_config", str(budget_path)]
eval_runs = []
run_identities = set()
if MODEL_KIND == "base":
    EVAL_STAGE = "stage0"
    MODEL_REVISION = cfg["model"]["revision"]
    eval_runs = [(None, cfg["model"]["repo"], "baseline_english")]
else:
    EVAL_STAGE = cfg["stage"]
    MODEL_REVISION = None
    epochs = int(cfg["training"]["num_train_epochs"])
    selected_epochs = [epochs]
    if INCLUDE_EARLIER_EPOCHS:
        selected_epochs.extend(range(1, epochs))
    checkpoint_root = Path(TRAIN_ROOT) / "checkpoints" / cfg["stage"] / cfg["run_name"]
    for epoch in selected_epochs:
        checkpoint = checkpoint_root / f"epoch_{epoch}"
        marker_path = checkpoint / "stage1_checkpoint.json"
        if not marker_path.is_file():
            raise FileNotFoundError(f"Missing saved SFT checkpoint marker: {marker_path}. Check TRAIN_ROOT.")
        marker = json.loads(marker_path.read_text())
        if marker["epoch"] != epoch:
            raise ValueError(f"Checkpoint epoch mismatch: {checkpoint}")
        run_identities.add(marker["run_identity"])
        run_name = cfg["run_name"] if epoch == epochs else f"{cfg['run_name']}_epoch{epoch}"
        eval_runs.append((epoch, checkpoint, run_name))
    if len(run_identities) != 1:
        raise ValueError("Selected epoch checkpoints must belong to the same training run")
result_root = profile_root(BASELINE_ROOT, EVAL_PROFILE, budget, sampling) / "full/results"
suite = json.loads((Path(CODE_ROOT) / "configs/english_eval_suite.json").read_text())
progress_totals = {name: entry["rows"] for name, entry in suite["datasets"].items()}

def evaluation_command(checkpoint, run_name, smoke=False):
    command = [str(Path(EVAL_ENV) / "bin/python"), "-m", "eval.run_batched_eval",
               "--model", str(checkpoint), "--stage", EVAL_STAGE, "--run_name", run_name,
               "--output_root", BASELINE_ROOT, "--profile", EVAL_PROFILE,
               "--execution_config", "configs/eval_execution.yaml"] + RUN_ARGS
    if MODEL_REVISION is not None:
        command += ["--revision", MODEL_REVISION]
    if smoke:
        command += ["--smoke"]
    return command

print("Selected model:", MODEL_KIND, "Sampling:", sampling)
print("Budget forcing:", budget or "off")
print("Benchmarks:", ", ".join(progress_totals))
for epoch, checkpoint, run_name in eval_runs:
    print(f"{checkpoint} -> {result_root / EVAL_STAGE / (run_name + '.json')}")
print("Resolved sampling config:", sampling_path)

## Optional smoke test — five problems
Evaluate one problem per benchmark using the Pro epoch-5 checkpoint.
Smoke results are saved separately from the full evaluation.

In [ ]:
RUN_SMOKE_EVAL = False
if RUN_SMOKE_EVAL:
    epoch, checkpoint, run_name = eval_runs[0]
    run_logged(evaluation_command(checkpoint, run_name, smoke=True), cwd=CODE_ROOT,
               log_path=Path(TRAIN_ROOT) / f"{run_name}_{EVAL_LABEL}_smoke_console.log",
               progress_totals={name: 1 for name in progress_totals})
else:
    print("Smoke evaluation is off. Enable RUN_SMOKE_EVAL to test five problems.")

## Full evaluation — all five benchmarks
Enable the checkbox to evaluate the selected model. SFT defaults to epoch 5, followed
by epochs 1–4 only when explicitly selected above. Base runs once. Completed problems
are saved durably and reused on resume with identical model/code/settings. Summary JSON,
per-problem journals, and raw generations are written to the printed result directory.

In [ ]:
RUN_STAGE1_EVAL = False # @param {type:"boolean"}
if RUN_STAGE1_EVAL:
    for epoch, checkpoint, run_name in eval_runs:
        run_logged(evaluation_command(checkpoint, run_name), cwd=CODE_ROOT,
                   log_path=Path(TRAIN_ROOT) / f"{run_name}_{EVAL_LABEL}_console.log",
                   progress_totals=progress_totals)
else:
    print("Evaluation is off. Enable RUN_STAGE1_EVAL to evaluate the selected model.")